In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import polars as pl
from Bio import SeqIO
from ete4 import Tree

sys.path.insert(0, "..")
from project import check_run_settings, get_all, log, tool, update_run_json

In [ ]:
ALL = get_all()
OVERWRITE = False  # True re-aligns every gene pair and allows changing the settings of an existing run
WORKERS = 32  # PRANK is single-threaded: one job per core

# Guide trees whose scale was never fitted (scale 1, no scale_source) are refused: PRANK would
# read their branch lengths as CDS rates, ~15x too short for IGRs. True lets them through on purpose.
ALLOW_UNSCALED = False

PRANK = tool("prank")
if not Path(PRANK).exists() and shutil.which(PRANK) is None:
    raise FileNotFoundError(
        f"PRANK not found at {PRANK}; check tools.prank in config.yaml"
    )

In [ ]:
def build_cmd(in_fasta, tree_file, out_prefix) -> list[str]:
    return [
        PRANK,
        f"-d={in_fasta}",
        f"-t={tree_file}",
        f"-o={out_prefix}",
        "-DNA",
        "+F",  # insertions are never matched later (recommended for DNA)
        "-once",  # use the given guide tree, don't re-estimate it
        "-shortnames",  # names are cut at the first space, so they match the tree tips
    ]


def run_prank(
    gp: str, igr_dir: Path, tree_dir: Path, run_dir: Path, logs_dir: Path
) -> tuple[str, str]:
    in_fasta = igr_dir / f"gp_{gp}.fasta"
    tree_file = tree_dir / f"gp_{gp}.nwk"
    out_fasta = run_dir / f"gp_{gp}_aligned.fasta"
    log_file = logs_dir / f"gp_{gp}_prank.log"

    if out_fasta.exists() and not OVERWRITE:
        return gp, "skipped"

    # PRANK calls MAFFT/Exonerate through the shell, using paths built from the output name and
    # the working directory. Gene-pair names contain shell characters (e.g. '|' in TL1|2), which
    # break those calls silently. So each job runs in a scratch folder with a safe name, on copies
    # of its inputs with plain names, and the result is moved to its real name afterwards.
    safe = (
        re.sub(r"[^A-Za-z0-9_.-]", "_", gp)
        + "_"
        + hashlib.md5(gp.encode()).hexdigest()[:8]
    )
    tmp_dir = run_dir / "tmp" / safe
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    shutil.copy(in_fasta, tmp_dir / "in.fasta")
    shutil.copy(tree_file, tmp_dir / "in.nwk")
    raw_out = tmp_dir / "out.best.fas"  # PRANK appends .best.fas to the prefix

    res = subprocess.run(
        build_cmd("in.fasta", "in.nwk", "out"),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=tmp_dir,
    )
    log_file.write_text(res.stdout)

    if res.returncode != 0 or not raw_out.exists():
        (logs_dir / f"gp_{gp}_prank_error.log").write_text(res.stdout)
        return gp, f"error (exit {res.returncode})"  # tmp_dir kept for debugging

    shutil.move(raw_out, out_fasta)  # only finished alignments get the final name
    shutil.rmtree(tmp_dir, ignore_errors=True)
    return gp, "ok"

In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p, cfg = info["paths"], info["config"]
        prank_run, guide_run = info["runs"]["prank"], info["runs"]["guide_trees"]
        igr_dir = p["igr_by_pair"]
        tree_dir = p["guide_trees"] / guide_run
        run_dir = p["alignments"] / prank_run
        logs_dir = run_dir / "logs"
        run_json = run_dir / "run.json"
        log(f"{g}/{subg} | prank run: {prank_run} | guide trees: {guide_run}")

        if not p["gene_pairs_filtered"].exists():
            log(
                f"  no {p['gene_pairs_filtered'].name}, run 02_gene_pairs first; skipping"
            )
            continue

        # --- guide trees: must exist, and be scaled unless ALLOW_UNSCALED
        tree_json = tree_dir / "run.json"
        if not tree_json.exists():
            raise FileNotFoundError(
                f"{tree_json} not found; run prank_tree/05_igr_tree with guide_trees: {guide_run}"
            )
        tree_info = json.loads(tree_json.read_text())
        scale, scale_source = (
            tree_info.get("scale"),
            tree_info.get("scale_source") or "",
        )
        if scale == 1 and not scale_source and not ALLOW_UNSCALED:
            raise RuntimeError(
                f"guide_trees/{guide_run} was never scaled (scale 1, no scale_source). Run the second "
                "05 pass with the fitted scale, or set ALLOW_UNSCALED = True."
            )
        log(f"  guide-tree scale: {scale} ({scale_source or 'not fitted'})")

        exclude = set(cfg.get("exclude_pairs") or [])
        template = build_cmd(
            "<igr_by_pair>/gp_{gp}.fasta",
            "<guide_trees>/gp_{gp}.nwk",
            "<run_dir>/gp_{gp}",
        )
        settings = {
            "guide_trees_run_name": guide_run,
            "command_template": " ".join(template),
            "excluded": sorted(exclude),
        }
        check_run_settings(run_json, settings, OVERWRITE)

        # --- every gene pair needs a FASTA and a tree whose tips are exactly its sequence IDs
        gp_df = pl.read_csv(p["gene_pairs_filtered"], separator="\t")
        listed = [str(gp) for gp in gp_df["gene_pair"]]
        unknown_excl = exclude - set(listed)
        if unknown_excl:
            log(
                f"  note: exclude_pairs has gene pairs not in {p['gene_pairs_filtered'].name}: {sorted(unknown_excl)}"
            )
        problems = []
        for gp in listed:
            if gp in exclude:
                continue
            fasta, tree_file = igr_dir / f"gp_{gp}.fasta", tree_dir / f"gp_{gp}.nwk"
            if not fasta.exists():
                problems.append(f"{gp}: missing {fasta.name}")
                continue
            if not tree_file.exists():
                problems.append(f"{gp}: missing tree {tree_file.name}")
                continue
            ids = {rec.id for rec in SeqIO.parse(fasta, "fasta")}
            tips = {
                leaf.name for leaf in Tree(tree_file.read_text(), parser=5).leaves()
            }
            if ids != tips:
                problems.append(
                    f"{gp}: FASTA and tree differ ({len(ids - tips)} only in FASTA, {len(tips - ids)} only in tree)"
                )
        if problems:
            log(f"  {len(problems)} of {len(listed)} gene pairs have problems:")
            for prob in problems:
                print(f"    {prob}")
            raise RuntimeError(f"{g}/{subg}: inputs do not match; nothing was aligned.")

        # largest jobs first, so the long ones don't end up running alone at the end
        size = dict(
            zip(
                gp_df["gene_pair"].cast(pl.String),
                (gp_df["igr_count"] * gp_df["mean_len"]).to_list(),
            )
        )
        pairs = sorted(
            (gp for gp in listed if gp not in exclude), key=lambda gp: -size[gp]
        )
        log(
            f"  {len(pairs)} gene pairs to align ({len(exclude & set(listed))} excluded) | {WORKERS} workers"
        )

        logs_dir.mkdir(parents=True, exist_ok=True)
        update_run_json(
            run_json,
            run_name=prank_run,
            group=g,
            subgroup=subg,
            status="running",
            date_started=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            date_finished=None,
            **settings,
            prank=PRANK,
            guide_tree_scale=scale,
            guide_tree_scale_source=scale_source,
            allow_unscaled=ALLOW_UNSCALED,
            iqtree_run_name=tree_info.get("iqtree_run_name"),
            mafft_run_name=tree_info.get("mafft_run_name"),
            cds_run_name=tree_info.get("cds_run_name"),
            input_fastas=str(igr_dir.relative_to(p["dir"])),
            gene_pairs_file=str(p["gene_pairs_filtered"].relative_to(p["dir"])),
            n_gene_pairs=len(pairs),
        )

        # --- align
        t0 = time.time()
        counts: dict[str, int] = {}
        errors = []
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            tasks = [
                executor.submit(run_prank, gp, igr_dir, tree_dir, run_dir, logs_dir)
                for gp in pairs
            ]
            for i, future in enumerate(as_completed(tasks), 1):
                gp, status = future.result()
                key = "error" if status.startswith("error") else status
                counts[key] = counts.get(key, 0) + 1
                if key == "error":
                    errors.append(f"{gp}: {status}")
                    log(f"  [ERROR] {gp}: {status}")
                if i % 10 == 0 or i == len(tasks):
                    log(f"    {i}/{len(tasks)} done | {counts}")

        runtime = time.time() - t0
        update_run_json(
            run_json,
            status="complete" if not errors else "complete_with_errors",
            date_finished=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            runtime_sec=round(runtime, 1),
            counts=counts,
            errors=errors,
        )

        stale = {
            f.name.removeprefix("gp_").removesuffix("_aligned.fasta")
            for f in run_dir.glob("gp_*_aligned.fasta")
        } - set(pairs)
        log(
            f"  {counts} | runtime {int(runtime // 3600)}h{int(runtime % 3600 // 60):02d}m -> {run_dir.relative_to(p['dir'])}/"
        )
        if stale:
            log(
                f"  note: {len(stale)} alignments in this run are for gene pairs not aligned now (excluded or no longer kept): {sorted(stale)}"
            )

log("Done")